# 自选股分类诊断

**目标**：给所有自选股打类型标签，确认分类逻辑是否合理。

分类结果满意后，`scripts/classifier.py` 会复用这里的逻辑写入数据库。

In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../scripts')

import db
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('✅ 环境加载完成')

In [ ]:
# ── 行业关键词映射 ────────────────────────────────────
FINANCIAL_KW  = ['银行', '保险', '券商', '信托', '证券', '基金', '期货', 'bank', 'insurance', 'financial']
UTILITY_KW    = ['电力', '水务', '燃气', '热力', '核电', 'utility', 'power', 'water', 'gas']
CYCLICAL_KW   = ['钢铁', '煤炭', '化工', '地产', '航运', '有色', '建材', '石油', '水泥',
                 'steel', 'coal', 'chemical', 'real estate', 'shipping']

def _pct_to_float(val):
    """把 '8.76%' 或 0.0876 统一转成 float"""
    if val is None: return None
    if isinstance(val, str):
        try: return float(val.strip('%'))
        except: return None
    return float(val)

def classify(code, market, fundamentals):
    """返回 (company_type, 判断理由)"""
    annual = fundamentals.get('annual', []) if fundamentals else []

    # 非A股数据不足，单独标注
    if market != 'cn' or not annual:
        return 'non_cn', f'市场={market.upper()}, 暂无财报数据'

    # 取最近3年ROE
    roe_vals = [_pct_to_float(y.get('roe')) for y in annual[:3]]
    roe_vals = [v for v in roe_vals if v is not None]
    roe_avg  = sum(roe_vals) / len(roe_vals) if roe_vals else None
    loss_years = sum(1 for v in roe_vals if v < 0)

    # 营收3年复合增速（如果有数据）
    revenues = [_pct_to_float(y.get('revenue')) for y in annual[:4] if y.get('revenue')]
    cagr = None
    if len(revenues) >= 2 and revenues[-1] and revenues[-1] > 0:
        cagr = ((revenues[0] / revenues[-1]) ** (1 / (len(revenues)-1)) - 1) * 100

    # 科创板前缀
    is_star = code.startswith('688') or code.startswith('301')

    # 1. 初创/亏损型（优先判断）
    if loss_years >= 2:
        return 'pre_profit', f'近{len(roe_vals)}年有{loss_years}年亏损，ROE均值={roe_avg:.1f}%'

    # 2. 成长科技型
    if is_star or (cagr and cagr > 20):
        reason = '科创板' if is_star else f'营收CAGR={cagr:.1f}%'
        return 'growth_tech', reason

    # 3. 成熟价值型（兜底）
    roe_str = f'ROE均值={roe_avg:.1f}%' if roe_avg else '数据不足'
    return 'mature_value', roe_str

print('✅ 分类函数定义完成')

In [ ]:
# ── 跑全部自选股 ──────────────────────────────────────
conn = db.get_conn()
stocks = conn.execute("""
    SELECT w.stock_code as code, w.status, p.market
    FROM user_watchlist w
    LEFT JOIN stock_prices p ON p.code = w.stock_code
    WHERE w.user_id = 1
    GROUP BY w.stock_code
""").fetchall()

rows = []
for s in stocks:
    code   = s['code']
    market = s['market'] or 'cn'
    f      = db.get_fundamentals(code)
    annual = f.get('annual', []) if f else []

    roe_vals = [_pct_to_float(y.get('roe')) for y in annual[:3]]
    roe_vals = [v for v in roe_vals if v is not None]
    roe_latest = roe_vals[0] if roe_vals else None

    company_type, reason = classify(code, market, f)

    rows.append({
        'code':         code,
        'market':       market,
        'status':       s['status'],
        'type':         company_type,
        'roe_latest':   roe_latest,
        'annual_years': len(annual),
        'reason':       reason,
    })

df = pd.DataFrame(rows)
print(f'共 {len(df)} 只股票')
df['type'].value_counts()

In [ ]:
# ── 按类型查看明细 ────────────────────────────────────
TYPE_LABELS = {
    'mature_value': '成熟价值型',
    'growth_tech':  '成长科技型',
    'financial':    '金融型',
    'cyclical':     '周期型',
    'utility':      '公用事业型',
    'pre_profit':   '初创亏损型',
    'non_cn':       '非A股（数据待补）',
}

for t, label in TYPE_LABELS.items():
    subset = df[df['type'] == t]
    if subset.empty: continue
    print(f'\n── {label}（{len(subset)}只）──')
    for _, r in subset.iterrows():
        roe_str = f"ROE={r['roe_latest']:.1f}%" if r['roe_latest'] is not None else 'ROE=N/A'
        print(f"  {r['code']}  {roe_str}  [{r['reason']}]")

In [ ]:
# ── ML 特征预览（为以后建模埋点）────────────────────────
# 这些字段将来是 ML 模型的 feature
feature_df = df[df['annual_years'] > 0][[
    'code', 'type', 'roe_latest'
]].dropna()

print('\n当前可用作 ML 特征的股票数量:', len(feature_df))
print('类型分布:')
print(feature_df['type'].value_counts())
print('\nROE 分布按类型:')
feature_df.groupby('type')['roe_latest'].describe().round(1)